# Version Comparison Analysis — v24 / v25 / v26 / v26-mle (D1 Sepsis)

Analyzes `results/version_comparison_D1.csv` produced by `version_comparison.py`
(7 miners incl. Flower, seed 42, 5×1000 shadow traces, vs variant-based 5-fold R1).

Sections:
1. Load & overview
2. Agreement with ground truth (Pearson / Spearman / MAE / spread)
3. Calibration scatter (the money plot)
4. Per-miner deviation from R1
5. Openness profile (regular vs mutated fitness)
6. Acceptance analysis + soundness normalization
7. Mutation-share stability
8. Roadmap for further analyses

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('results/version_comparison_D1.csv')
if 'Seed' in df.columns:
    df = df[df.Seed == 42].copy()  # multi-seed CSV: analyze the official seed
MINERS_NF = ['Alpha', 'Alpha+', 'Heuristics', 'Heuristics_Strict',
             'Inductive_Infrequent', 'Inductive_Strict']
VERSIONS = ['v24', 'v25', 'v26', 'v26-mle']
R1 = df[df.Version == 'v24'].set_index('Miner')['R1']

pivot = df.pivot(index='Miner', columns='Version', values='Gen')
pivot['R1'] = R1
pivot.loc[MINERS_NF + ['Flower'], ['R1'] + VERSIONS]

## 2. Agreement with ground truth

Three criteria, three different questions (Method_GenShadow.md §5):
**Spearman** = same idea (ordering), **MAE** = trustworthy absolute values,
**spread** = discriminative power. Flower excluded (litmus checked separately).

In [ ]:
def pearson(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    return np.corrcoef(x, y)[0, 1] if x.std() > 0 and y.std() > 0 else np.nan

def spearman(x, y):
    return pearson(np.argsort(np.argsort(x)), np.argsort(np.argsort(y)))

y = R1[MINERS_NF].values
rows = []
for v in VERSIONS:
    sub = df[df.Version == v].set_index('Miner')
    for col, name in [('Gen', 'fitness'), ('Accept', 'accept')]:
        if sub.loc[MINERS_NF, col].isna().any():
            continue
        x = sub.loc[MINERS_NF, col].values.astype(float)
        rows.append({'Version': v, 'Score': name,
                     'Pearson': round(pearson(x, y), 3),
                     'Spearman': round(spearman(x, y), 3),
                     'MAE': round(np.mean(np.abs(x - y)), 3),
                     'Spread': round(x.max() - x.min(), 3),
                     'Flower': sub.loc['Flower', col]})
pd.DataFrame(rows)

## 3. Calibration scatter — the money plot

Perfect calibration = points on the diagonal. Watch v26-mle hug the line.
This is the figure that belongs in the report once D2+ confirms the pattern.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
markers = {'v24': 'o', 'v25': 's', 'v26': '^', 'v26-mle': 'D'}
for v in VERSIONS:
    sub = df[df.Version == v].set_index('Miner')
    ax.scatter(R1[MINERS_NF], sub.loc[MINERS_NF, 'Gen'],
               marker=markers[v], s=70, alpha=0.8, label=v)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect calibration')
for m in MINERS_NF:
    ax.annotate(m, (R1[m], df[(df.Version == 'v26-mle') & (df.Miner == m)]['Gen'].iloc[0]),
                fontsize=7, xytext=(4, -8), textcoords='offset points')
ax.set_xlabel('R1 — variant-based 5-fold CV fitness (ground truth)')
ax.set_ylabel('Gen_shadow (mean fitness)')
ax.set_title('Calibration vs ground truth, D1 Sepsis (flower excluded)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Per-miner deviation from R1

Where does each version over/under-shoot? v24/v25 are systematically strict on
Alpha+ and the Heuristics pair; mle closes most of that gap.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.2
xpos = np.arange(len(MINERS_NF))
for i, v in enumerate(VERSIONS):
    sub = df[df.Version == v].set_index('Miner')
    dev = sub.loc[MINERS_NF, 'Gen'].values.astype(float) - R1[MINERS_NF].values
    ax.bar(xpos + (i - 1.5) * width, dev, width, label=v)
ax.axhline(0, color='k', lw=1)
ax.set_xticks(xpos)
ax.set_xticklabels(MINERS_NF, rotation=20, ha='right')
ax.set_ylabel('Gen − R1')
ax.set_title('Deviation from ground truth per miner')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Openness profile — regular vs mutated shadow traces

Under the Katz-consistent proposal (v25+), mutated traces should be *plausible*,
so a well-generalizing model replays both groups similarly. A large gap means the
model penalizes novel events specifically (Heuristics behavior on BPI 2017).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, v in zip(axes, ['v25', 'v26', 'v26-mle']):
    sub = df[df.Version == v].set_index('Miner')
    reg = sub.loc[MINERS_NF, 'Fit_Regular'].values.astype(float)
    mut = sub.loc[MINERS_NF, 'Fit_Mutated'].values.astype(float)
    xpos = np.arange(len(MINERS_NF))
    ax.bar(xpos - 0.18, reg, 0.36, label='regular')
    ax.bar(xpos + 0.18, mut, 0.36, label='mutated')
    ax.set_xticks(xpos)
    ax.set_xticklabels(MINERS_NF, rotation=35, ha='right', fontsize=8)
    ax.set_title(v)
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('mean trace fitness')
axes[0].legend()
plt.suptitle('Openness profile: fitness on regular vs mutated shadow traces')
plt.tight_layout()
plt.show()

## 6. Acceptance analysis + soundness normalization

Raw acceptance conflates generalization with the net class's ability to replay
*anything* perfectly (Heuristics nets accept ≈ nothing, even original traces).
Fix: normalize by the model's perfect-replay rate **on the original log**.

⚠️ The next cell rediscovers all 7 models and replays the full log — takes ~1–2 min.

In [ ]:
import pm4py
from pm4py.algo.conformance.tokenreplay import algorithm as token_replay
from pm4py.objects.petri_net.obj import PetriNet, Marking
from pm4py.objects.petri_net.utils import petri_utils

log = pm4py.read_xes('../data/Sepsis Cases - Event Log_1_all/Sepsis Cases - Event Log.xes.gz')
log = pm4py.convert_to_event_log(log)

def flower_miner(l):
    net = PetriNet('Flower Model'); p = PetriNet.Place('mid'); net.places.add(p)
    for act in set(e['concept:name'] for t in l for e in t):
        tr = PetriNet.Transition(f't_{act}', act); net.transitions.add(tr)
        petri_utils.add_arc_from_to(p, tr, net); petri_utils.add_arc_from_to(tr, p, net)
    im, fm = Marking(), Marking(); im[p] = 1; fm[p] = 1
    return net, im, fm

MINER_FNS = {
    'Alpha': pm4py.discover_petri_net_alpha,
    'Alpha+': pm4py.discover_petri_net_alpha_plus,
    'Heuristics': pm4py.discover_petri_net_heuristics,
    'Heuristics_Strict': lambda l: pm4py.discover_petri_net_heuristics(l, dependency_threshold=0.99),
    'Inductive_Strict': lambda l: pm4py.discover_petri_net_inductive(l, noise_threshold=0.0),
    'Inductive_Infrequent': lambda l: pm4py.discover_petri_net_inductive(l, noise_threshold=0.2),
    'Flower': flower_miner,
}

acc_orig = {}
for name, fn in MINER_FNS.items():
    net, im, fm = fn(log)
    replayed = token_replay.apply(log, net, im, fm)
    acc_orig[name] = np.mean([1.0 if r['trace_is_fit'] else 0.0 for r in replayed])
    print(f'{name:22s} perfect-replay rate on ORIGINAL log: {acc_orig[name]:.4f}')

In [ ]:
rows = []
for v in ['v26', 'v26-mle']:
    sub = df[df.Version == v].set_index('Miner')
    for m in MINERS_NF + ['Flower']:
        raw = float(sub.loc[m, 'Accept'])
        base = acc_orig[m]
        rows.append({'Version': v, 'Miner': m, 'Accept_raw': round(raw, 4),
                     'Accept_origlog': round(base, 4),
                     'Accept_normalized': round(min(raw / base, 1.0), 4) if base > 0 else np.nan})
acc_df = pd.DataFrame(rows)
display(acc_df.pivot(index='Miner', columns='Version',
                     values=['Accept_raw', 'Accept_normalized']))

# Agreement of normalized acceptance with R1 (excl. Flower, excl. NaN)
for v in ['v26', 'v26-mle']:
    s = acc_df[(acc_df.Version == v) & (acc_df.Miner != 'Flower')].dropna(subset=['Accept_normalized'])
    if len(s) >= 3:
        x = s['Accept_normalized'].values
        yy = R1[s['Miner']].values
        print(f'{v}: normalized-acceptance vs R1  Pearson={pearson(x, yy):.3f}  '
              f'Spearman={spearman(x, yy):.3f}')

## 7. Mutation-share stability across iterations

In [ ]:
sub = df[(df.Version == 'v26') & (df.Miner == 'Inductive_Strict')]
mut_counts = ast.literal_eval(sub['Mut_Traces'].iloc[0])
print(f'Mutated traces per iteration (of 1000): {mut_counts}')
print(f'Share: {np.mean(mut_counts)/10:.1f}% ± {np.std(mut_counts)/10:.1f}%')
print('\nCompare: BPI 2017 at N=6 had ~3.8% — the mutation mass is dataset-dependent,')
print('driven by per-context Good–Turing mass at the effective (backed-off) order.')

## 8. Roadmap — further analyses worth adding here

1. **Seed sensitivity**: rerun `version_comparison.py` with seeds 1–5, plot the
   distribution of MAE/Pearson per version. Confirms the v26-mle win is not a
   single-seed artifact (cheap: ~10 min per seed).
2. **Acceptance ground truth**: extend `r1_demo.py` to also record the fraction of
   held-out traces *perfectly* replayed (R1-accept). Then validate `gen_accept`
   against R1-accept instead of R1-fitness — the apples-to-apples comparison.
3. **Generator validation** (the big one, Method_GenShadow.md §7): build the
   generator from 80% of variants, measure (a) what fraction of held-out variants
   it can produce (generator recall of the real future), (b) n-gram distribution
   distance between shadow log and held-out fold. Directly tests the premise
   "shadow ≈ future valid behavior" — no baseline method validates itself this way.
4. **Probe-distribution study**: shadow-trace length distribution vs original vs
   held-out; variant-novelty histogram (edit distance to nearest log variant).
   Shows *how far* from observed behavior the probe reaches per mode (log vs mle).
5. **D2 replication**: rerun this notebook on BPI 2013 Incidents (TLRA 0.80) —
   watch `Dup_Kept` there; it is the first dataset where the dedup retry cap
   might actually bite.
6. **Sensitivity to `num_shadow_traces` / `iterations`**: convergence plot of the
   score std — justifies the 5×1000 protocol in the report.